In [1]:
import pandas as pd
from taxaplease import TaxaPlease
import os
import sys

In [2]:
#f = pd.read_csv("~/shared-team/mike/gpha-mscape-orangebox-virus-reclassification/results/test_climbid_test_runid_kraken_results.txt", sep="\t")
df1 = pd.read_csv("~/shared-team/mike/gpha-mscape-orangebox-virus-reclassification/results/test_climbid_test_runid_kraken_report.txt", sep="\t",
                 names=[
                         "percent_coverage",
                         "no_reads_covered",
                         "no_reads_assigned",
                         "rank",
                         "taxid",
                         "name"
                     ])

In [3]:
taxaPlease = TaxaPlease()
taxaPlease.isVirus()

TypeError: TaxaPlease.isVirus() missing 1 required positional argument: 'inputTaxid'

In [4]:
test = df1.copy()
l1 = []
for x in df1["taxid"]:
    a = taxaPlease.isVirus(x)
    l1.append(a)
    #taxaPlease.isVirus(df1["taxid"][0:2])
test = test.assign(is_virus=l1)
test

,percent_coverage,no_reads_covered,no_reads_assigned,rank,taxid,name,is_virus
0,0.01,17,17,U,0,unclassified,False
1,99.99,181588,0,R,1,root,False
2,99.99,181588,0,R1,10239,Viruses,True
3,99.76,181164,0,R2,2732004,Varidnaviria,True
4,99.76,181164,0,K,2732005,Bamfordvirae,True
...,...,...,...,...,...,...,...
92,0.00,1,0,F2,2960224,unclassified Ensavirinae,True
93,0.00,1,0,G,12059,Enterovirus,True
94,0.00,1,0,S,3428501,Enterovirus alpharhino,True
95,0.00,1,0,S1,147711,Rhinovirus A,True


In [5]:
test = df1.copy()
test = test.assign(is_virus=test['taxid'].apply(lambda x: taxaPlease.isVirus(x)))
test

,percent_coverage,no_reads_covered,no_reads_assigned,rank,taxid,name,is_virus
0,0.01,17,17,U,0,unclassified,False
1,99.99,181588,0,R,1,root,False
2,99.99,181588,0,R1,10239,Viruses,True
3,99.76,181164,0,R2,2732004,Varidnaviria,True
4,99.76,181164,0,K,2732005,Bamfordvirae,True
...,...,...,...,...,...,...,...
92,0.00,1,0,F2,2960224,unclassified Ensavirinae,True
93,0.00,1,0,G,12059,Enterovirus,True
94,0.00,1,0,S,3428501,Enterovirus alpharhino,True
95,0.00,1,0,S1,147711,Rhinovirus A,True


In [6]:
def is_virus(df:pd.DataFrame) -> pd.DataFrame:
    """initiate TaxaPlease and use isVirus function (->bool) to determine if taxids found are listed as viruses in NCBI database."""
    taxaPlease = TaxaPlease()
    df = df.assign(is_virus=df['taxid'].apply(lambda x: taxaPlease.isVirus(x)))
    return df

df1 = is_virus(df1)
df1

,percent_coverage,no_reads_covered,no_reads_assigned,rank,taxid,name,is_virus
0,0.01,17,17,U,0,unclassified,False
1,99.99,181588,0,R,1,root,False
2,99.99,181588,0,R1,10239,Viruses,True
3,99.76,181164,0,R2,2732004,Varidnaviria,True
4,99.76,181164,0,K,2732005,Bamfordvirae,True
...,...,...,...,...,...,...,...
92,0.00,1,0,F2,2960224,unclassified Ensavirinae,True
93,0.00,1,0,G,12059,Enterovirus,True
94,0.00,1,0,S,3428501,Enterovirus alpharhino,True
95,0.00,1,0,S1,147711,Rhinovirus A,True


In [7]:
df_viral_species = pd.read_csv("~/shared-team/mike/gpha-mscape-orangebox-virus-reclassification/results/midpoint.csv")

def load_results(in_path:str) ->pd.DataFrame:
    df_res = pd.read_csv(in_path,
                         sep="\t",
                         names=[
                             "isClassified",
                             "header",
                             "taxid",
                             "length",
                             "kmer_map"
                         ]
                        )
    return df_res

df_kmer = load_results("~/shared-team/mike/gpha-mscape-orangebox-virus-reclassification/results/test_climbid_test_runid_kraken_results.txt")

In [8]:
df_viral_species

,Unnamed: 0,percent_coverage,no_reads_covered,no_reads_assigned,rank,taxid,name,is_virus
0,11,57.46,104350,0,S,3241411,Mastadenovirus caesari,True
1,20,42.23,76691,0,S,3241406,Mastadenovirus blackbeardi,True
2,39,0.07,122,0,S,3241402,Mastadenovirus adami,True
3,51,0.11,197,0,S,3052557,Orthorubulavirus laryngo...,True
4,53,0.01,16,0,S,3052556,Orthorubulavirus hominis,True
5,58,0.06,109,0,S,3049953,Respirovirus pneumoniae,True
6,60,0.01,21,0,S,3049952,Respirovirus laryngotrac...,True
7,65,0.03,62,0,S,3049954,Orthopneumovirus hominis,True
8,77,0.00,6,0,S,3433758,Betacoronavirus hongko...,True
9,79,0.00,3,0,S,3433757,Betacoronavirus graved...,True


In [9]:
def get_kmer_matches(df_filtered_report:pd.DataFrame, df_results:pd.DataFrame, climb_id:str, run_id:str) -> pd.DataFrame:
    """for climb_id/run_id that return species level match(es) create dataframe of data where kmer matches for viral species
    occur. This is to be used in the info json."""
    list_taxid = list(set(df_results.taxid))
    print(list_taxid)
    #df_match = [lambda x: df_results.taxid.isin(x) for x in list_taxid]
    #df_match = pd.DataFrame([lambda x: df_filtered_report.taxid.isin(list_taxid)]).value_counts()
    df_match = df_filtered_report[df_filtered_report["taxid"].isin(list_taxid)]
    return df_match

In [10]:
test = get_kmer_matches(df_kmer, df_viral_species, "test", test)

[3049952, 3049953, 3049954, 3241411, 3052556, 3052557, 3433809, 3428501, 3241402, 3433756, 3433757, 3241406, 3433758]


In [11]:
test

,isClassified,header,taxid,length,kmer_map


In [67]:
test = df_kmer.taxid.astype(str).str.contains("3049952")
test.value_counts()

length
False    181605
Name: count, dtype: int64

In [74]:
df_kmer.kmer_map

0         0:93 10509:5 129951:1 10509:5 129951:5 10509:1...
1         0:98 129951:5 0:5 129951:27 0:61 129951:20 0:3...
2         0:164 10509:12 0:39 45659:6 108098:7 45659:5 0...
3         0:101 108098:51 0:21 108098:83 45659:3 108098:...
4         0:158 108098:26 0:22 108098:5 0:25 108098:5 0:...
                                ...                        
181600    0:129 129951:79 0:32 129951:24 0:24 129951:2 0...
181601    0:148 129951:26 0:11 129951:2 0:6 129951:5 0:2...
181602    0:155 129951:2 0:32 129951:2 0:1 129951:1 0:10...
181603    0:113 129951:2 10509:3 129951:12 0:95 129951:5...
181604    0:196 45659:29 108098:5 45659:3 108098:19 0:7 ...
Name: kmer_map, Length: 181605, dtype: object

In [12]:
kmer_map

NameError: name 'kmer_map' is not defined

In [ ]:
test = 